In [12]:
import os
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import random

In [13]:
reproduciblity = True
if reproduciblity:
    random.seed(10)

In [14]:
import numpy as np
import pandas as pd


def feature_extractor_m3(data, time_resolution):
    """
    Feature extraction for M3 competition datasets.

    Parameters
    ----------
    data : pd.DataFrame
        DataFrame containing at least column ['y'].
        The index can be timestamp/year/month/etc.
    time_resolution : str
        One of: "Monthly", "Quarterly", "Yearly"

    Returns
    -------
    pd.DataFrame
        DataFrame with lag and rolling features.
    """

    data = data.copy()

    # Basic autoregressive lags
    data.loc[:, "y-1"] = data.loc[:, "y"].shift(1)
    data.loc[:, "y-2"] = data.loc[:, "y"].shift(2)
    data.loc[:, "y-3"] = data.loc[:, "y"].shift(3)

    if time_resolution == "monthly":

        # Seasonal lags for monthly data
        data.loc[:, "y-6"] = data.loc[:, "y"].shift(6)
        data.loc[:, "y-12"] = data.loc[:, "y"].shift(12)
        data.loc[:, "y-24"] = data.loc[:, "y"].shift(24)

        # Rolling features from y-1
        data.loc[:, "y-1_rolling_mean_3"] = data.loc[:, "y-1"].rolling(3).mean()
        data.loc[:, "y-1_rolling_std_3"] = data.loc[:, "y-1"].rolling(3).std()

        data.loc[:, "y-1_rolling_mean_6"] = data.loc[:, "y-1"].rolling(6).mean()
        data.loc[:, "y-1_rolling_std_6"] = data.loc[:, "y-1"].rolling(6).std()

        data.loc[:, "y-1_rolling_mean_12"] = data.loc[:, "y-1"].rolling(12).mean()
        data.loc[:, "y-1_rolling_std_12"] = data.loc[:, "y-1"].rolling(12).std()

        # Rolling features from seasonal lag y-12
        data.loc[:, "y-12_rolling_mean_3"] = data.loc[:, "y-12"].rolling(3).mean()
        data.loc[:, "y-12_rolling_std_3"] = data.loc[:, "y-12"].rolling(3).std()

        data.loc[:, "y-12_rolling_mean_6"] = data.loc[:, "y-12"].rolling(6).mean()
        data.loc[:, "y-12_rolling_std_6"] = data.loc[:, "y-12"].rolling(6).std()

        data.loc[:, "y-12_rolling_mean_12"] = data.loc[:, "y-12"].rolling(12).mean()
        data.loc[:, "y-12_rolling_std_12"] = data.loc[:, "y-12"].rolling(12).std()

    elif time_resolution == "quarterly":

        # Seasonal lags for quarterly data
        data.loc[:, "y-4"] = data.loc[:, "y"].shift(4)
        data.loc[:, "y-8"] = data.loc[:, "y"].shift(8)

        # Rolling features from y-1
        data.loc[:, "y-1_rolling_mean_2"] = data.loc[:, "y-1"].rolling(2).mean()
        data.loc[:, "y-1_rolling_std_2"] = data.loc[:, "y-1"].rolling(2).std()

        data.loc[:, "y-1_rolling_mean_4"] = data.loc[:, "y-1"].rolling(4).mean()
        data.loc[:, "y-1_rolling_std_4"] = data.loc[:, "y-1"].rolling(4).std()

        data.loc[:, "y-1_rolling_mean_8"] = data.loc[:, "y-1"].rolling(8).mean()
        data.loc[:, "y-1_rolling_std_8"] = data.loc[:, "y-1"].rolling(8).std()

        # Rolling features from seasonal lag y-4
        data.loc[:, "y-4_rolling_mean_2"] = data.loc[:, "y-4"].rolling(2).mean()
        data.loc[:, "y-4_rolling_std_2"] = data.loc[:, "y-4"].rolling(2).std()

        data.loc[:, "y-4_rolling_mean_4"] = data.loc[:, "y-4"].rolling(4).mean()
        data.loc[:, "y-4_rolling_std_4"] = data.loc[:, "y-4"].rolling(4).std()

    elif time_resolution == "yearly":

        # Yearly data has no fixed intra-year seasonality
        data.loc[:, "y-4"] = data.loc[:, "y"].shift(4)
        data.loc[:, "y-5"] = data.loc[:, "y"].shift(5)

        # Rolling features from recent yearly values
        data.loc[:, "y-1_rolling_mean_2"] = data.loc[:, "y-1"].rolling(2).mean()
        data.loc[:, "y-1_rolling_std_2"] = data.loc[:, "y-1"].rolling(2).std()

        data.loc[:, "y-1_rolling_mean_3"] = data.loc[:, "y-1"].rolling(3).mean()
        data.loc[:, "y-1_rolling_std_3"] = data.loc[:, "y-1"].rolling(3).std()

        data.loc[:, "y-1_rolling_mean_5"] = data.loc[:, "y-1"].rolling(5).mean()
        data.loc[:, "y-1_rolling_std_5"] = data.loc[:, "y-1"].rolling(5).std()

        # Rolling features from y-2
        data.loc[:, "y-2_rolling_mean_2"] = data.loc[:, "y-2"].rolling(2).mean()
        data.loc[:, "y-2_rolling_std_2"] = data.loc[:, "y-2"].rolling(2).std()

        data.loc[:, "y-2_rolling_mean_3"] = data.loc[:, "y-2"].rolling(3).mean()
        data.loc[:, "y-2_rolling_std_3"] = data.loc[:, "y-2"].rolling(3).std()

    else:
        raise ValueError(
            "time_resolution must be one of: 'Monthly', 'Quarterly', 'Yearly'"
        )

    # Remove rows with missing values caused by shift and rolling operations
    data = data.dropna()

    # Reset index like your M3 code
    data.index = np.arange(len(data))

    return data

In [15]:
m3s = {"monthly": 18, "quarterly": 8, "yearly": 6}

M3_TYPE = list(m3s.keys())[0]  # Change this to "quarterly" or "yearly" as needed
TEST_SIZE = m3s[M3_TYPE]
df = pd.read_csv(f"data/M3/M3_{M3_TYPE}_TSTS.csv")
os.mkdir(f"data/M3/Extracted/{M3_TYPE}/")

M3_TYPE
TEST_SIZE

18

In [16]:
# Make sure data is ordered correctly inside each time series
df = df.sort_values(["series_id", "timestamp"])
# Get unique series ids
unique_ids = df["series_id"].unique()
len(unique_ids)

1428

In [17]:
n_series = min(1000, len(unique_ids))

random_ids = pd.Series(unique_ids).sample(
    n=n_series,
    random_state=42
).tolist()
len(random_ids)

1000

In [18]:
# Filter dataframe
sample_df = df[df["series_id"].isin(random_ids)]
sample_df.head()

,series_id,category,value,timestamp
0,M1,MICRO,2640.0,1990-01
1,M1,MICRO,2640.0,1990-02
2,M1,MICRO,2160.0,1990-03
3,M1,MICRO,4200.0,1990-04
4,M1,MICRO,3360.0,1990-05


In [19]:
length_dict = {}
for series_id, group in sample_df.groupby("series_id"):
    print(f"Series ID: {series_id}, Length: {len(group)}")
    if len(group) in length_dict:
        length_dict[len(group)] += 1
    else:
        length_dict[len(group)] = 1
length_dict

Series ID: M1, Length: 68
Series ID: M100, Length: 69
Series ID: M1000, Length: 134
Series ID: M1002, Length: 134
Series ID: M1003, Length: 134
Series ID: M1004, Length: 134
Series ID: M1006, Length: 134
Series ID: M1007, Length: 134
Series ID: M1008, Length: 134
Series ID: M1009, Length: 134
Series ID: M1014, Length: 134
Series ID: M1019, Length: 134
Series ID: M102, Length: 69
Series ID: M1020, Length: 134
Series ID: M1022, Length: 134
Series ID: M1023, Length: 134
Series ID: M1024, Length: 134
Series ID: M1025, Length: 134
Series ID: M1026, Length: 134
Series ID: M1027, Length: 134
Series ID: M1028, Length: 134
Series ID: M1030, Length: 134
Series ID: M1032, Length: 134
Series ID: M1033, Length: 134
Series ID: M1035, Length: 134
Series ID: M1036, Length: 86
Series ID: M1037, Length: 134
Series ID: M1038, Length: 134
Series ID: M1039, Length: 134
Series ID: M1040, Length: 134
Series ID: M1041, Length: 134
Series ID: M1042, Length: 134
Series ID: M1044, Length: 134
Series ID: M1045, L

{68: 14,
 69: 179,
 134: 236,
 86: 1,
 142: 6,
 133: 74,
 144: 157,
 132: 19,
 108: 6,
 129: 1,
 84: 2,
 66: 2,
 120: 8,
 107: 2,
 143: 7,
 74: 3,
 110: 3,
 115: 1,
 100: 1,
 81: 2,
 104: 1,
 83: 2,
 99: 1,
 89: 1,
 76: 1,
 94: 1,
 78: 1,
 73: 1,
 70: 2,
 135: 51,
 71: 20,
 96: 20,
 72: 7,
 138: 4,
 126: 134,
 141: 8,
 140: 3,
 136: 2,
 139: 6,
 98: 4,
 122: 1,
 121: 5}

In [20]:
sample_series_ids = list(sample_df["series_id"].unique())
len(sample_series_ids)

1000

In [21]:
for series_id in sample_series_ids:
    series_data = sample_df[sample_df["series_id"] == series_id]
    series_data = series_data[["timestamp", "value"]]
    series_data = series_data.set_index("timestamp")
    series_data = series_data.rename(columns={"value": "y"})
    series_data = feature_extractor_m3(series_data, time_resolution=M3_TYPE)
    series_data.to_csv(f"data/M3/Extracted/{M3_TYPE}/{series_id}.csv", index=False)

